In [80]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [81]:

data_path = Path("../Data/raw/census.csv")

if not data_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {data_path.resolve()}")

df_original = pd.read_csv(data_path)
df = df_original.copy(deep=True)

print("Dataset path:", data_path.resolve())
print("Loaded dataset shape:", df.shape)

Dataset path: D:\DEPI\Projects\ML\ML_P2\Data\raw\census.csv
Loaded dataset shape: (45222, 14)


In [82]:
display(df.head())
print("shape", df.shape)
print("columns", df.columns.tolist())
df.info()
display(df.dtypes.astype(str).to_frame("dtype"))

,age,workclass,education_level,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=50K
1,50,Self-emp-not-inc,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,13.0,United-States,<=50K
2,38,Private,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=50K
3,53,Private,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,40.0,United-States,<=50K
4,28,Private,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,<=50K


shape (45222, 14)
columns ['age', 'workclass', 'education_level', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45222 entries, 0 to 45221
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              45222 non-null  int64  
 1   workclass        45222 non-null  object 
 2   education_level  45222 non-null  object 
 3   education-num    45222 non-null  float64
 4   marital-status   45222 non-null  object 
 5   occupation       45222 non-null  object 
 6   relationship     45222 non-null  object 
 7   race             45222 non-null  object 
 8   sex              45222 non-null  object 
 9   capital-gain     45222 non-null  float64
 10  capital-loss     45222 non-null  float64
 11  hours-per-week   45222 non-null  float64
 12  native-country   4

,dtype
age,int64
workclass,object
education_level,object
education-num,float64
marital-status,object
occupation,object
relationship,object
race,object
sex,object
capital-gain,float64


In [83]:
display(df.describe().T)
display(df.describe(include=["object"]).T)

,count,mean,std,min,25%,50%,75%,max
age,45222.0,38.547941,13.217870,17.0,28.0,37.0,47.0,90.0
education-num,45222.0,10.118460,2.552881,1.0,9.0,10.0,13.0,16.0
capital-gain,45222.0,1101.430344,7506.430084,0.0,0.0,0.0,0.0,99999.0
capital-loss,45222.0,88.595418,404.956092,0.0,0.0,0.0,0.0,4356.0
hours-per-week,45222.0,40.938017,12.007508,1.0,40.0,40.0,45.0,99.0


,count,unique,top,freq
workclass,45222,7,Private,33307
education_level,45222,16,HS-grad,14783
marital-status,45222,7,Married-civ-spouse,21055
occupation,45222,14,Craft-repair,6020
relationship,45222,6,Husband,18666
race,45222,5,White,38903
sex,45222,2,Male,30527
native-country,45222,41,United-States,41292
income,45222,2,<=50K,34014


In [84]:
income_counts = df["income"].value_counts(dropna=False)
income_proportions = df["income"].value_counts(normalize=True, dropna=False)
display(income_counts.to_frame("count"))
display(income_proportions.to_frame("proportion"))

,count
income,
<=50K,34014
>50K,11208


,proportion
income,
<=50K,0.752156
>50K,0.247844


In [85]:
text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
missing_counts = df.isna().sum()
question_mark_counts = {column: int(df[column].astype("string").str.strip().eq("?").sum()) for column in text_columns}
whitespace_only_counts = {column: int(df[column].astype("string").str.fullmatch(r"\s*").fillna(False).sum()) for column in text_columns}
display(missing_counts.to_frame("missing_count"))
display(pd.Series(question_mark_counts, name="question_mark_count").to_frame())
display(pd.Series(whitespace_only_counts, name="whitespace_only_count").to_frame())

,missing_count
age,0
workclass,0
education_level,0
education-num,0
marital-status,0
occupation,0
relationship,0
race,0
sex,0
capital-gain,0


,question_mark_count
workclass,0
education_level,0
marital-status,0
occupation,0
relationship,0
race,0
sex,0
native-country,0
income,0


,whitespace_only_count
workclass,0
education_level,0
marital-status,0
occupation,0
relationship,0
race,0
sex,0
native-country,0
income,0


In [86]:
duplicate_count = int(df.duplicated().sum())
print("duplicate_rows", duplicate_count)

duplicate_rows 5982


In [87]:
expected_numeric_columns = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
numeric_conversion_errors = {}
for column in expected_numeric_columns:
    converted = pd.to_numeric(df[column], errors="coerce")
    numeric_conversion_errors[column] = int((converted.isna() & df[column].notna()).sum())
expected_columns = ["age", "workclass", "education_level", "education-num", "marital-status", "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]
print("numeric_conversion_errors", numeric_conversion_errors)
print("unexpected_columns", [column for column in df.columns if column not in expected_columns])
print("missing_expected_columns", [column for column in expected_columns if column not in df.columns])
display(df.dtypes.astype(str).to_frame("dtype"))

numeric_conversion_errors {'age': 0, 'education-num': 0, 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 0}
unexpected_columns []
missing_expected_columns []


,dtype
age,int64
workclass,object
education_level,object
education-num,float64
marital-status,object
occupation,object
relationship,object
race,object
sex,object
capital-gain,float64


In [88]:
categorical_columns = df.select_dtypes(include=["object", "string", "category", "bool"]).columns.tolist()
outer_whitespace_counts = {column: int(df[column].map(lambda value: isinstance(value, str) and value != value.strip()).sum()) for column in categorical_columns}
case_variants = {}
for column in categorical_columns:
    values = df[column].dropna().astype(str).unique().tolist()
    groups = {}
    for value in values:
        groups.setdefault(value.strip().casefold(), set()).add(value.strip())
    variants = {key: sorted(group) for key, group in groups.items() if len(group) > 1}
    if variants:
        case_variants[column] = variants
print("outer_whitespace_counts", outer_whitespace_counts)
print("case_variants", case_variants)
for column in categorical_columns:
    print(column, "unique_values", int(df[column].nunique(dropna=False)))
    display(df[column].value_counts(dropna=False).to_frame("count"))

outer_whitespace_counts {'workclass': 45222, 'education_level': 45222, 'marital-status': 45222, 'occupation': 45222, 'relationship': 45222, 'race': 45222, 'sex': 45222, 'native-country': 45222, 'income': 0}
case_variants {}
workclass unique_values 7


,count
workclass,
Private,33307
Self-emp-not-inc,3796
Local-gov,3100
State-gov,1946
Self-emp-inc,1646
Federal-gov,1406
Without-pay,21


education_level unique_values 16


,count
education_level,
HS-grad,14783
Some-college,9899
Bachelors,7570
Masters,2514
Assoc-voc,1959
11th,1619
Assoc-acdm,1507
10th,1223
7th-8th,823


marital-status unique_values 7


...

,count
sex,
Male,30527
Female,14695


native-country unique_values 41


,count
native-country,
United-States,41292
Mexico,903
Philippines,283
Germany,193
Puerto-Rico,175
Canada,163
El-Salvador,147
India,147
Cuba,133


income unique_values 2


,count
income,
<=50K,34014
>50K,11208


In [89]:
valid_numeric_ranges = {"age": (0, 120), "education-num": (1, 16), "capital-gain": (0, None), "capital-loss": (0, None), "hours-per-week": (0, 168)}
invalid_numeric_counts = {}
for column, (lower, upper) in valid_numeric_ranges.items():
    values = pd.to_numeric(df[column], errors="coerce")
    invalid = values.notna()
    if lower is not None:
        invalid &= values < lower
    if upper is not None:
        invalid |= values.notna() & (values > upper)
    invalid_numeric_counts[column] = int(invalid.sum())
valid_income_values = {"<=50K", ">50K"}
invalid_income_values = sorted(set(df["income"].dropna().astype(str).str.strip()) - valid_income_values)
print("invalid_numeric_counts", invalid_numeric_counts)
print("invalid_income_values", invalid_income_values)
print("zero_capital_gain", int(df["capital-gain"].eq(0).sum()))
print("zero_capital_loss", int(df["capital-loss"].eq(0).sum()))

invalid_numeric_counts {'age': 0, 'education-num': 0, 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 0}
invalid_income_values []
zero_capital_gain 41432
zero_capital_loss 43082


In [90]:
outlier_results = {}
for column in df.select_dtypes(include=np.number).columns:
    values = df[column].dropna()
    first_quartile = values.quantile(0.25)
    third_quartile = values.quantile(0.75)
    interquartile_range = third_quartile - first_quartile
    lower_bound = first_quartile - 1.5 * interquartile_range
    upper_bound = third_quartile + 1.5 * interquartile_range
    outlier_count = int(((values < lower_bound) | (values > upper_bound)).sum())
    outlier_results[column] = {"lower_bound": float(lower_bound), "upper_bound": float(upper_bound), "outlier_count": outlier_count}
display(pd.DataFrame.from_dict(outlier_results, orient="index"))

,lower_bound,upper_bound,outlier_count
age,-0.5,75.5,269
education-num,3.0,19.0,294
capital-gain,0.0,0.0,3790
capital-loss,0.0,0.0,2140
hours-per-week,32.5,52.5,11899


In [91]:
df_clean = df.copy(deep=True)
for column in df_clean.select_dtypes(include=["object", "string"]).columns:
    normalized_values = df_clean[column].astype("string").str.strip()
    df_clean[column] = normalized_values.mask(normalized_values.eq("") | normalized_values.eq("?"), pd.NA)
target_missing_count = int(df_clean["income"].isna().sum())
if target_missing_count:
    df_clean = df_clean.loc[df_clean["income"].notna()].copy()
for column in df_clean.select_dtypes(include=np.number).columns:
    if df_clean[column].isna().any():
        median_value = df_clean[column].median()
        if pd.isna(median_value):
            raise ValueError(f"No observed values available to impute {column}")
        df_clean[column] = df_clean[column].fillna(median_value)
for column in df_clean.select_dtypes(include=["object", "string"]).columns:
    if df_clean[column].isna().any():
        modes = df_clean[column].mode(dropna=True)
        if modes.empty:
            raise ValueError(f"No observed category available to impute {column}")
        df_clean[column] = df_clean[column].fillna(modes.iloc[0])
print("target_rows_removed_for_missing_target", target_missing_count)
print("remaining_missing_values", int(df_clean.isna().sum().sum()))

target_rows_removed_for_missing_target 0
remaining_missing_values 0


In [92]:
identifier_columns = [column for column in df_clean.columns if column.casefold() == "id" or column.casefold().endswith("_id")]
duplicate_count_after_missing_handling = int(df_clean.duplicated().sum())
print("identifier_columns", identifier_columns)
print("duplicate_rows_reviewed", duplicate_count_after_missing_handling)
print("duplicate_rows_removed", 0)
print("duplicate_rows_retained", duplicate_count_after_missing_handling)

identifier_columns []
duplicate_rows_reviewed 5982
duplicate_rows_removed 0
duplicate_rows_retained 5982


In [93]:
for column in df_clean.select_dtypes(include=["object", "string"]).columns:
    df_clean[column] = df_clean[column].astype("string").str.strip()
print("outer_whitespace_remaining", {column: int(df_clean[column].map(lambda value: isinstance(value, str) and value != value.strip()).sum()) for column in df_clean.select_dtypes(include=["object", "string"]).columns})
print("income_values", sorted(df_clean["income"].dropna().unique().tolist()))

outer_whitespace_remaining {'workclass': 0, 'education_level': 0, 'marital-status': 0, 'occupation': 0, 'relationship': 0, 'race': 0, 'sex': 0, 'native-country': 0, 'income': 0}
income_values ['<=50K', '>50K']


In [94]:
if not set(df_clean["income"].dropna().unique()).issubset({"<=50K", ">50K"}):
    raise ValueError("Unexpected income values need review before cleaning can continue")
invalid_replacements = {}
for column, (lower, upper) in valid_numeric_ranges.items():
    values = pd.to_numeric(df_clean[column], errors="coerce")
    invalid = values.notna()
    if lower is not None:
        invalid &= values < lower
    if upper is not None:
        invalid |= values.notna() & (values > upper)
    invalid_replacements[column] = int(invalid.sum())
    if invalid.any():
        df_clean.loc[invalid, column] = np.nan
        median_value = pd.to_numeric(df_clean[column], errors="coerce").median()
        if pd.isna(median_value):
            raise ValueError(f"No valid values available to replace invalid values in {column}")
        df_clean[column] = pd.to_numeric(df_clean[column], errors="coerce").fillna(median_value)
print("impossible_numeric_values_replaced", invalid_replacements)

impossible_numeric_values_replaced {'age': 0, 'education-num': 0, 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 0}


In [95]:
print("iqr_outliers_reviewed", {column: values["outlier_count"] for column, values in outlier_results.items()})
print("outliers_removed", 0)
print("outliers_capped_or_clipped", 0)

iqr_outliers_reviewed {'age': 269, 'education-num': 294, 'capital-gain': 3790, 'capital-loss': 2140, 'hours-per-week': 11899}
outliers_removed 0
outliers_capped_or_clipped 0


In [96]:
df_clean.columns = df_clean.columns.str.strip()
if df_clean.columns.duplicated().any():
    raise ValueError("Duplicate column names after trimming")
for column in expected_numeric_columns:
    df_clean[column] = pd.to_numeric(df_clean[column], errors="raise")
print("columns", df_clean.columns.tolist())
print("dtypes", df_clean.dtypes.astype(str).to_dict())
print("shape", df_clean.shape)

columns ['age', 'workclass', 'education_level', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
dtypes {'age': 'int64', 'workclass': 'string', 'education_level': 'string', 'education-num': 'float64', 'marital-status': 'string', 'occupation': 'string', 'relationship': 'string', 'race': 'string', 'sex': 'string', 'capital-gain': 'float64', 'capital-loss': 'float64', 'hours-per-week': 'float64', 'native-country': 'string', 'income': 'string'}
shape (45222, 14)


In [97]:
comparison = pd.DataFrame({"original_dtype": df_original.dtypes.astype(str), "cleaned_dtype": df_clean.dtypes.astype(str), "original_missing": df_original.isna().sum(), "cleaned_missing": df_clean.isna().sum()})
display(comparison)
print("original_shape", df_original.shape)
print("cleaned_shape", df_clean.shape)
print("original_duplicate_rows", int(df_original.duplicated().sum()))
print("cleaned_duplicate_rows", int(df_clean.duplicated().sum()))
print("original_missing_values", int(df_original.isna().sum().sum()))
print("cleaned_missing_values", int(df_clean.isna().sum().sum()))

,original_dtype,cleaned_dtype,original_missing,cleaned_missing
age,int64,int64,0,0
workclass,object,string,0,0
education_level,object,string,0,0
education-num,float64,float64,0,0
marital-status,object,string,0,0
occupation,object,string,0,0
relationship,object,string,0,0
race,object,string,0,0
sex,object,string,0,0
capital-gain,float64,float64,0,0


original_shape (45222, 14)
cleaned_shape (45222, 14)
original_duplicate_rows 5982
cleaned_duplicate_rows 5982
original_missing_values 0
cleaned_missing_values 0


In [98]:
assert df_clean.columns.tolist() == df_original.columns.tolist()
assert not df_clean.isna().any().any()
assert not df_clean.columns.duplicated().any()
assert set(df_clean["income"].unique()).issubset({"<=50K", ">50K"})
assert all(int(df_clean[column].map(lambda value: isinstance(value, str) and value != value.strip()).sum()) == 0 for column in df_clean.select_dtypes(include=["object", "string"]).columns)
for column, (lower, upper) in valid_numeric_ranges.items():
    assert (lower is None or df_clean[column].ge(lower).all())
    assert (upper is None or df_clean[column].le(upper).all())
print("final_shape", df_clean.shape)
print("final_missing_values", int(df_clean.isna().sum().sum()))
print("final_duplicate_rows", int(df_clean.duplicated().sum()))
print("final_income_values", sorted(df_clean["income"].unique().tolist()))

final_shape (45222, 14)
final_missing_values 0
final_duplicate_rows 5982
final_income_values ['<=50K', '>50K']


In [99]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45222 entries, 0 to 45221
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              45222 non-null  int64  
 1   workclass        45222 non-null  string 
 2   education_level  45222 non-null  string 
 3   education-num    45222 non-null  float64
 4   marital-status   45222 non-null  string 
 5   occupation       45222 non-null  string 
 6   relationship     45222 non-null  string 
 7   race             45222 non-null  string 
 8   sex              45222 non-null  string 
 9   capital-gain     45222 non-null  float64
 10  capital-loss     45222 non-null  float64
 11  hours-per-week   45222 non-null  float64
 12  native-country   45222 non-null  string 
 13  income           45222 non-null  string 
dtypes: float64(4), int64(1), string(9)
memory usage: 4.8 MB


In [100]:
from pathlib import Path

output_dir = Path("../Data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "census_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print("Saved:", output_path.resolve())
print("Dataset shape:", df_clean.shape)

Saved: D:\DEPI\Projects\ML\ML_P2\Data\processed\census_cleaned.csv
Dataset shape: (45222, 14)


In [101]:
# cleaned_path = data_path.with_name("census_cleaned.csv")
# if cleaned_path.exists():
#     existing_cleaned = pd.read_csv(cleaned_path)
#     pd.testing.assert_frame_equal(df_clean, existing_cleaned, check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)
#     print("existing_cleaned_file_matches", cleaned_path)
# else:
#     df_clean.to_csv(cleaned_path, index=False)
#     print("saved_path", cleaned_path)
# print("saved_shape", pd.read_csv(cleaned_path).shape)